# Week2-3: 10D MCMC from scratch

Now we have the position of all the stars in the 1x1 degree on the sky. \
We assume that each star could belong ot the dwarf galaxy that follows a Gaussian distribution or a the MW forground that follows a flat distribution. \
Now we will determine the 2D distribution of dwarf galaxies on the sky, as well as the member probability of each star. 



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import corner
from glob import glob
import os

In [3]:
data_path = '../data/Dwarfs/'
res_path = '../result/'
fig_path = '../figure/'

files = glob(os.path.join(data_path, "Dwarf*"))
files.sort()
print (np.array(files))



['../data/Dwarfs/Dwarf_00' '../data/Dwarfs/Dwarf_01'
 '../data/Dwarfs/Dwarf_02' '../data/Dwarfs/Dwarf_03'
 '../data/Dwarfs/Dwarf_04' '../data/Dwarfs/Dwarf_05'
 '../data/Dwarfs/Dwarf_06' '../data/Dwarfs/Dwarf_07'
 '../data/Dwarfs/Dwarf_08' '../data/Dwarfs/Dwarf_09'
 '../data/Dwarfs/Dwarf_10' '../data/Dwarfs/Dwarf_11'
 '../data/Dwarfs/Dwarf_12' '../data/Dwarfs/Dwarf_13'
 '../data/Dwarfs/Dwarf_14' '../data/Dwarfs/Dwarf_15'
 '../data/Dwarfs/Dwarf_16' '../data/Dwarfs/Dwarf_17'
 '../data/Dwarfs/Dwarf_18' '../data/Dwarfs/Dwarf_19']


In [4]:
def new_coords(x, y, alpha):

    x_new = x * np.cos(alpha) + y * np.sin(alpha)
    y_new = - x * np.sin(alpha) + y * np.cos(alpha)

    return x_new, y_new

# define the two axis of the ellipse using sigma_x, sigma_y, and alpha (change within pi/2)

def log_Likelihood(theta, x, y, vr, e_vr):

    xc, yc, sigma_x, sigma_y, alpha, mvr_dg, dvr_dg, mvr_mw, dvr_mw, f = theta

    x1, y1 = new_coords(x, y, alpha)
    xc1, yc1 = new_coords(xc, yc, alpha)

    sigma_v_dg = np.sqrt(dvr_dg**2 + e_vr**2)
    sigma_v_mw = np.sqrt(dvr_mw**2 + e_vr**2)

    # Calculate the likelihood of being in the dwarf galaxy
    L_dg_xy = 1/(2*np.pi*sigma_x * sigma_y)*np.exp(-0.5*((x1-xc1)**2./sigma_x**2.+(y1-yc1)**2./sigma_y**2.))
    L_dg_vr = 1/(2*np.pi*sigma_v_dg)*np.exp(-0.5*((vr-mvr_dg)**2./sigma_v_dg**2))

    L_dg = L_dg_xy * L_dg_vr
    # Calculate the likelihood of being in the MW foreground
    L_mw_xy = 1/10000**2.
    L_mw_vr = 1/(2*np.pi*sigma_v_mw)*np.exp(-0.5*((vr-mvr_mw)**2./sigma_v_mw**2))
    L_mw = L_mw_xy * L_mw_vr
    
    logL = np.sum(np.log(f*L_dg + (1-f)*L_mw))

    return logL


def prob_member(theta, x, y, vr, e_vr ):

    xc, yc, sigma_x, sigma_y, alpha, mvr_dg, dvr_dg, mvr_mw, dvr_mw, f  = theta

    x1, y1 = new_coords(x, y, alpha)
    xc1, yc1 = new_coords(xc, yc, alpha)

    sigma_v_dg = np.sqrt(dvr_dg**2 + e_vr**2)
    sigma_v_mw = np.sqrt(dvr_mw**2 + e_vr**2)

    P_dg_xy = 1/(2*np.pi*sigma_x * sigma_y)*np.exp(-0.5*((x1-xc1)**2./sigma_x**2.+(y1-yc1)**2./sigma_y**2.))
    P_dg_vr = 1/(2*np.pi*sigma_v_dg)*np.exp(-0.5*((vr-mvr_dg)**2./sigma_v_dg**2))

    P_dg = P_dg_xy * P_dg_vr

    P_mw_xy = 1/10000**2.
    P_mw_vr = 1/(2*np.pi*sigma_v_mw)*np.exp(-0.5*((vr-mvr_mw)**2./sigma_v_mw**2))
    P_mw = P_mw_xy * P_mw_vr
    
    
    prob =P_dg / (P_dg + P_mw)

    return prob


In [5]:
# dwarf_list = np.array([13, 16, 17, 18, 19])
dwarf_list = np.arange(2, 10, 1)

In [6]:
for k in dwarf_list:

    df = pd.read_csv(data_path+'Dwarf_%02d'%k)
    ind_novr = df['vr'] > 990
    df.loc[ind_novr, 'vr'] = 0
    df.loc[ind_novr, 'd_vr'] = 10000    


In [7]:
for k in dwarf_list:

    df = pd.read_csv(files[k])


    
    xc0 = 0
    yc0 = 0
    sigma_x0 = 100
    sigma_y0 = 100
    alpha0 = 0
    mvr_dg0 = 0.0
    dvr_dg0 = 10.0
    mvr_mw0 = 0
    dvr_mw0 = 50.0
    f0=0.1

    param0 = xc0, yc0, sigma_x0, sigma_y0, alpha0, mvr_dg0, dvr_dg0, mvr_mw0, dvr_mw0, f0


    logL0 = log_Likelihood(param0, df['X_pc'], df['Y_pc'], df['vr'], df['d_vr'])

    # refine the following steps based on the uncertainties of the results
    step_xc = 10
    step_yc = 20
    step_sigma_x = 10
    step_sigma_y = 20
    step_alpha = np.pi/60
    step_mvr_dg = 1
    step_dvr_dg = .5
    step_mvr_mw = 1
    step_dvr_mw = .5
    step_f = 0.001
    nsteps = 100000

    samples = np.zeros((nsteps, 12))
    accept = 1


    alpha_min = 0
    alpha_max = np.pi/2

    if k in [0, 10, 11, 15, 17, 18]:

        alpha_min = -np.pi/4
        alpha_max = np.pi/4

    for i in range(nsteps):

        param1 = param0
        logL1 = logL0

        samples[i, :10] = param1
        samples[i, 10] = logL1
        samples[i, 11] = accept
        accept = 0

        xc1, yc1, sigma_x1, sigma_y1, alpha1, mvr_dg1, dvr_dg1, mvr_mw1, dvr_mw1, f1 = param1
        xc1 = xc1 + np.random.normal(0, step_xc)   
        yc1 = yc1 + np.random.normal(0, step_yc)
        sigma_x1 = sigma_x1 + np.random.normal(0, step_sigma_x)
        sigma_y1 = sigma_y1 + np.random.normal(0, step_sigma_y)
        alpha1 = alpha1 + np.random.normal(0, step_alpha)
    

        mvr_dg1 = mvr_dg1 + np.random.normal(0, step_mvr_dg)
        dvr_dg1 = dvr_dg1 + np.random.normal(0, step_dvr_dg)   
        mvr_mw1 = mvr_mw1 + np.random.normal(0, step_mvr_mw)
        dvr_mw1 = dvr_mw1 + np.random.normal(0, step_dvr_mw)
        f1 = f1 + np.random.normal(0, step_f)


        if all([
            0 < f1 < 1,
            0 < sigma_x1 < 1000,
            0 < sigma_y1 < 1000,
            alpha_min <= alpha1 < alpha_max,
            -300 < mvr_dg1 < 300,
            -100 < mvr_mw1 < 100,
            0 < dvr_dg1 < 30,
            20 < dvr_mw1 < 60,
            ]):
    
            param1 = xc1, yc1, sigma_x1, sigma_y1, alpha1, mvr_dg1, dvr_dg1, mvr_mw1, dvr_mw1, f1
            logL1 = log_Likelihood(param1,  df['X_pc'], df['Y_pc'], df['vr'], df['d_vr'])           

            if logL1 > logL0:

                param0 = param1
                logL0 = logL1
                accept = 1

            else:

                a = np.random.uniform(0, 1)
                if (logL1-logL0) > np.log(a):

                    param0 = param1
                    logL0 = logL1
                    accept = 1

    ind_accept = samples[:, -1] == 1
    print ("Dwarf%02d acceptance rate: %.2f"%(k, len(samples[ind_accept, -1])/nsteps))
    samples =  samples[int(nsteps*0.1):]
    np.savetxt(res_path+'Dwarf_%02d_10D_MCMC_chain.txt'%k, samples)

    fig = corner.corner(
    samples[:,:10],
    labels=[r"xc (pc)", r"yc (pc)", r"$\sigma_x$ (pc)", r"$\sigma_y$ (pc)", r"$\alpha$", r"$\mu_{dg}$ (km/s)", r"$\sigma_{dg}$ (km/s)", r"$\mu_{mw}$ (km/s)", r"$\sigma_{mw}$ (km/s)", r"$f$"],
    show_titles=True,
    title_fmt=".3f",
    quantiles=[0.16, 0.5, 0.84],
    levels=(0.16, 0.5, 0.84),
    color="royalblue",
    fill_contours=True)
    plt.savefig(fig_path+'Dwarf_%02d_10D_MCMC_corner_new.png'%k, dpi=300)
    plt.close(fig)
    
    p_member = np.zeros(len(df))
    indices = np.random.choice(len(samples), 1000, replace=False)
    draw_samples = samples[indices]

    for i in range(len(draw_samples)):

        param = draw_samples[i,:10]

        p_member += prob_member(param, df['X_pc'], df['Y_pc'], df['vr'], df['d_vr'])

    p_member = p_member/len(draw_samples)
    df['P_member'] = p_member
    df.to_csv(res_path+'Dwarf_%02d_10D_MCMC_prob.csv'%k,index=False)

    fig=plt.figure()
    plt.hist(p_member, bins=50);
    plt.xlabel("Probability of being a member of Dwarf%02d"%1)
    plt.savefig(fig_path+'Dwarf_%02d_10D_MCMC_prob_new.png'%k, dpi=300)
    plt.close(fig)

    fig=plt.figure()
    im=plt.scatter(df['X_pc'], df['Y_pc'], c=df['P_member'], s=5, cmap='plasma')
    cb=plt.colorbar(im)
    cb.set_label("membership probability")
    plt.xlabel("X (pc)")
    plt.ylabel("Y (pc)")
    plt.savefig(fig_path+'Dwarf_%02d_10D_MCMC_prob_XY_new.png'%k, dpi=300)
    plt.close(fig)

    break



Dwarf02 acceptance rate: 0.01
